# Lineage Graphs, Dependency Tracing, and Impact Analysis Lab


## Purpose

This lab traces downstream dependencies from a source dataset to affected artifacts.

In [ ]:
import pandas as pd

entities = pd.DataFrame([
    {"entity_id": "D_source_A", "entity_type": "source_dataset", "version": "2026-01"},
    {"entity_id": "D_source_B", "entity_type": "source_dataset", "version": "2026-01"},
    {"entity_id": "D_joined_1", "entity_type": "derived_dataset", "version": "v1"},
    {"entity_id": "F_features_2", "entity_type": "feature_table", "version": "v2"},
    {"entity_id": "M_model_3", "entity_type": "model", "version": "v3"},
    {"entity_id": "R_eval_3", "entity_type": "evaluation_report", "version": "v3"},
])

activities = pd.DataFrame([
    {"activity_id": "A_join", "activity_type": "join"},
    {"activity_id": "A_feature", "activity_type": "feature_engineering"},
    {"activity_id": "A_train", "activity_type": "model_training"},
    {"activity_id": "A_eval", "activity_type": "evaluation"},
])

agents = pd.DataFrame([
    {"agent_id": "G_data_team", "agent_type": "team"},
    {"agent_id": "G_ml_team", "agent_type": "team"},
    {"agent_id": "G_governance", "agent_type": "team"},
])

relations = pd.DataFrame([
    {"source": "D_source_A", "relation": "used_by", "target": "A_join"},
    {"source": "D_source_B", "relation": "used_by", "target": "A_join"},
    {"source": "A_join", "relation": "generated", "target": "D_joined_1"},
    {"source": "D_joined_1", "relation": "used_by", "target": "A_feature"},
    {"source": "A_feature", "relation": "generated", "target": "F_features_2"},
    {"source": "F_features_2", "relation": "used_by", "target": "A_train"},
    {"source": "A_train", "relation": "generated", "target": "M_model_3"},
    {"source": "M_model_3", "relation": "used_by", "target": "A_eval"},
    {"source": "A_eval", "relation": "generated", "target": "R_eval_3"},
])

entities, activities, agents, relations.head()

In [ ]:
def downstream_dependencies(start_entity: str, relation_table: pd.DataFrame) -> set[str]:
    visited = set()
    frontier = [start_entity]

    while frontier:
        current = frontier.pop()

        if current in visited:
            continue

        visited.add(current)

        children = relation_table.loc[relation_table["source"] == current, "target"].tolist()
        frontier.extend(children)

    visited.remove(start_entity)
    return visited

impacted_by_source_b = downstream_dependencies("D_source_B", relations)

impact_table = pd.DataFrame([{
    "source_entity": "D_source_B",
    "impacted_nodes": ", ".join(sorted(impacted_by_source_b)),
    "model_impacted": "M_model_3" in impacted_by_source_b,
    "evaluation_impacted": "R_eval_3" in impacted_by_source_b,
}])

impact_table

## Interpretation

Impact analysis identifies which downstream artifacts must be reviewed when a source dataset changes or contains defects.